# Hướng Dẫn Giải Thích Chi Tiết: `src/data_loader.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của tệp `src/data_loader.py`. File này đóng vai trò cực kỳ quan trọng trong khâu chuẩn bị dữ liệu đầu vào cho toàn bộ hệ thống dự báo.

---

## 🔍 1. Tổng Quan Về Các Thư Viện Sử Dụng

Trong `data_loader.py`, chúng ta nhập các thư viện chính:
- `yfinance as yf`: Tải dữ liệu chứng khoán trực tuyến từ Yahoo Finance.
- `pandas as pd` và `numpy as np`: Xử lý và tính toán bảng dữ liệu.
- `pandas_ta as ta`: Thư viện tính toán các chỉ báo phân tích kỹ thuật.
- `urllib.request`: Gửi yêu cầu HTTP đến DNSE API để tải dữ liệu lịch sử của VNM.

In [ ]:
import sys
import os
# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))

from src.data_loader import fetch_and_prepare_data, format_vn
print("Import thành công!")

## ⚙️ 2. Công Dụng Của Các Hàm Trong `data_loader.py`

### Hàm `format_vn(value)`
- **Mục đích:** Định dạng một số thực thành chuỗi hiển thị tiền tệ VNĐ (ngăn cách hàng nghìn bằng dấu chấm và thêm đuôi VNĐ).
- **Ví dụ:** `format_vn(65000.5)` -> `"65.000,50 VNĐ"`.

In [ ]:
price = 15427452.12
print(f"Giá gốc: {price}")
print(f"Sau định dạng: {format_vn(price)}")

### Hàm `fetch_and_prepare_data(ticker, start_date, end_date)`

Đây là hàm trái tim của khâu chuẩn bị dữ liệu, thực hiện tuần tự các bước sau:

#### Bước A: Tải dữ liệu từ Yahoo Finance
Tải dữ liệu trực tuyến bằng `yf.download`. Nếu là mã cổ phiếu nước ngoài (`GOOGL`, `META`), hàm tự động quy đổi giá trị (`open`, `high`, `low`, `close`, `adj close`) sang VNĐ bằng cách nhân với tỷ giá cố định `25.400`.

#### Bước B: Tải bổ sung dữ liệu lịch sử trước 2019 của Vinamilk
Dữ liệu trường chỉ bắt đầu từ năm 2019. Để huấn luyện Transformer sâu hơn, hàm sử dụng DNSE Chart API để tải thêm dữ liệu từ năm 2012. API sử dụng Epoch Timestamp để yêu cầu dữ liệu:
`https://services.entrade.com.vn/chart-api/v2/ohlcs/stock?from={start_epoch}&to={end_epoch}&symbol=VNM&resolution=1D`

#### Bước C: Hợp nhất và loại bỏ trùng lặp
Dữ liệu được sắp xếp theo thời gian tăng dần, loại bỏ các ngày trùng lặp giữa các nguồn bằng hàm `.drop_duplicates(subset=['date'])`.

#### Bước D: Tính toán các chỉ báo kỹ thuật (Feature Engineering)
Hàm tính toán các chỉ báo thông qua thư viện `pandas_ta`:
1. **SMA (Simple Moving Average):** Trung bình động đơn giản (SMA_10, SMA_30).
2. **EMA (Exponential Moving Average):** Trung bình động lũy thừa (EMA_14) - phản ứng nhanh hơn với biến động gần nhất.
3. **RSI (Relative Strength Index):** Chỉ số sức mạnh tương đối (RSI_14) để đo tình trạng quá mua/quá bán.
4. **MACD (Moving Average Convergence Divergence):** Đo lường xu hướng động lượng (MACD_12_26_9).
5. **Bollinger Bands:** Dải đo biến động (BBL_5_2.0, BBM_5_2.0, BBU_5_2.0).
6. **ROC (Rate of Change):** Tốc độ thay đổi giá trong 10 phiên gần nhất (ROC_10).
7. **ADX (Average Directional Index):** Đo cường độ của xu hướng (ADX_14).

#### Bước E: Xử lý dữ liệu khuyết thiếu
Sử dụng phương pháp điền ngược (`bfill`) và điền xuôi (`ffill`) để loại bỏ các giá trị NaN sinh ra do quá trình tính toán chỉ báo ở những dòng đầu tiên.

In [ ]:
# Thử tải dữ liệu mẫu cho Vinamilk và hiển thị bảng dữ liệu
df_sample = fetch_and_prepare_data("VNM.VN", start_date="2023-01-01", end_date="2023-06-01")
print(f"Kích thước dữ liệu mẫu: {df_sample.shape}")
print("\nCác cột dữ liệu và chỉ báo kỹ thuật đã tạo:")
print(df_sample.columns.tolist())

# Hiển thị 5 dòng đầu tiên
df_sample[['date', 'open', 'close', 'ema_14', 'rsi_14', 'adx_14']].head()

## 📈 3. Trực Quan Hóa Chỉ Báo Đã Tính Toán

Hãy vẽ biểu đồ giá đóng cửa kèm theo đường EMA_14 và chỉ báo ADX_14 vừa được sinh ra.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(14, 7))

# Biểu đồ giá và EMA
plt.subplot(2, 1, 1)
plt.plot(df_sample['date'], df_sample['close'], label='Giá Đóng Cửa (VNĐ)', color='blue')
plt.plot(df_sample['date'], df_sample['ema_14'], label='EMA 14 Phiên', color='orange', linestyle='--')
plt.title('Giá Cổ Phiếu VNM & Chỉ Báo EMA 14')
plt.legend()
plt.grid(True)

# Biểu đồ ADX (Độ mạnh xu hướng)
plt.subplot(2, 1, 2)
plt.plot(df_sample['date'], df_sample['adx_14'], label='ADX 14', color='purple')
plt.axhline(25, color='red', linestyle=':', label='Mốc Xu Hướng Mạnh (>25)')
plt.title('Chỉ Báo Độ Mạnh Xu Hướng (ADX 14)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()